In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# from torchmetrics import Accuracy

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
import glob

### Parameters

In [84]:
n_input = 20
n_hidden = 512
n_out = 5
batch_size = 100
learning_rate = 0.01
agent_n_epochs = 500
adversary_n_epochs = 500

### Data

In [85]:
# find all pickle files in the current directory using glob

path = "/Users/ens/Library/Mobile Documents/com~apple~CloudDocs/Rupali/new"
all_files = glob.glob(path + "/*.pkl")

# create a list of dataframes
li = []

for filename in all_files:
    df = pd.read_pickle(filename)
    li.append(df)

# concatenate the list of dataframes into one dataframe
data = pd.concat(li, axis=0, ignore_index=True)

In [86]:
# save observation column of arrays to a numpy array
obs = np.array(data['obs'].to_list())
obs.shape

(1000000, 20)

In [87]:
act = np.array(data['actions'].to_list())
act.shape

(1000000,)

In [88]:
ids = np.array(data['agent_index'].to_list())
ids.shape

(1000000,)

In [89]:
# create a pytorch dataloader from the obervation and action arrays
# the dataloader will be used to train the neural network
# the dataloader will return a batch of observations and actions
# the batch size is set to 100
# the shuffle parameter is set to True so that the data is shuffled before each epoch

dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs).float(), torch.from_numpy(act).float())
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

## Functions

In [90]:
def data_prep(data):

    # do train test validation split on the data
    # the data is split into 80% train, 10% test, 10% validation
    # the data is shuffled before splitting

    train, test, val = np.split(data.sample(frac=1), [int(.8*len(data)), int(.9*len(data))])

    print("train shape: ", train.shape)
    print("test shape: ", test.shape)
    print("val shape: ", val.shape)

    # 

    obs_train, obs_test, obs_val = np.array(train['obs'].to_list()), np.array(test['obs'].to_list()), np.array(val['obs'].to_list())
    act_train, act_test, act_val = np.array(train['actions'].to_list()), np.array(test['actions'].to_list()), np.array(val['actions'].to_list())
    ids_train, ids_test, ids_val = np.array(train['agent_index'].to_list()), np.array(test['agent_index'].to_list()), np.array(val['agent_index'].to_list())

    # create test, train and validation dataloaders
    # the dataloaders will be used to train the neural network
    # the dataloaders will return a batch of observations and actions
    # the batch size is set to 100
    # the shuffle parameter is set to True so that the data is shuffled before each epoch

    train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_train), torch.from_numpy(act_train))
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


    test_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_test), torch.from_numpy(act_test))
    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    val_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_val), torch.from_numpy(act_val))
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    return train_dataloader, test_dataloader, val_dataloader

In [91]:
class PolicyNetwork(nn.Module):
    def __init__(self):
        super(PolicyNetwork, self).__init__()
        self.model = nn.Sequential(nn.Linear(n_input, n_hidden, bias=True),
                      nn.Tanh(),
                      nn.Linear(n_hidden, n_hidden, bias=True),
                      nn.Tanh(),
                      nn.Linear(n_hidden, n_hidden, bias=True),
                      nn.Tanh(),
                      nn.Linear(n_hidden, n_out, bias=True))

    def forward(self, x):
        logits = self.model(x)
        return logits

In [92]:
def policy_model():

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using {device} device")
    
    model = PolicyNetwork().to(device)
    print(model)

    loss_function = nn.CrossEntropyLoss()  
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    return model, loss_function, optimizer

In [93]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [94]:
policy, loss_fn, optimizer = policy_model()

# get dataloaders for train, test and validation data
train_dataloader, test_dataloader, val_dataloader = data_prep(data)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, policy, loss_fn, optimizer)
    test_loop(test_dataloader, policy, loss_fn)
print("Done!")

Using cpu device
PolicyNetwork(
  (model): Sequential(
    (0): Linear(in_features=20, out_features=512, bias=True)
    (1): Tanh()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): Tanh()
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): Tanh()
    (6): Linear(in_features=512, out_features=5, bias=True)
  )
)
train shape:  (800000, 20)
test shape:  (100000, 20)
val shape:  (100000, 20)
Epoch 1
-------------------------------
loss: 1.600708  [    0/800000]
loss: 1.471909  [10000/800000]
loss: 1.506314  [20000/800000]
loss: 1.386125  [30000/800000]
loss: 1.357573  [40000/800000]
loss: 1.373757  [50000/800000]
loss: 1.454596  [60000/800000]
loss: 1.330050  [70000/800000]
loss: 1.495825  [80000/800000]
loss: 1.358549  [90000/800000]
loss: 1.430428  [100000/800000]
loss: 1.400793  [110000/800000]
loss: 1.372122  [120000/800000]
loss: 1.492017  [130000/800000]
loss: 1.314610  [140000/800000]
loss: 1.312934  [150000/800000]
loss: 1.397045  [160000/80

In [ ]:
def plotting(train_x_axis, train_losses, train_accuracies, test_x_axis, test_losses, test_accuracies):
    plt.plot(test_x_axis, test_losses, label = 'Testing')
    plt.plot(train_x_axis, train_losses, label = 'Training')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend()
    plt.title("Learning rate %f"%(learning_rate))
    plt.show()


    plt.plot(test_x_axis, test_accuracies , label = 'Testing')
    plt.plot(train_x_axis, train_accuracies, label = 'Training')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend()
    plt.title("Learning rate %f"%(learning_rate))
    plt.show()   

## AGENT

In [ ]:
agent_df = df[df['agent_index']==3]
agent_df = agent_df.drop(columns=['obs', 'agent_index'])

print(np.shape(agent_df))

In [ ]:
agent_train_X, agent_test_X, agent_train_Y, agent_test_Y = data_prep(agent_df)

In [ ]:
agent_model, agent_loss_function, agent_optimizer = policy_model()

In [ ]:
agent_train_x_axis, agent_train_losses, agent_train_accuracies, agent_test_x_axis, agent_test_losses, agent_test_accuracies = training_and_testing(agent_model, agent_n_epochs, agent_train_X, agent_train_Y, agent_test_X, agent_test_Y, agent_loss_function, agent_optimizer)

In [ ]:
plotting(agent_train_x_axis, agent_train_losses, agent_train_accuracies, agent_test_x_axis, agent_test_losses, agent_test_accuracies)